In [3]:
# Import libraries
import csv

import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42

An error occurred: module 'importlib.metadata' has no attribute 'packages_distributions'


/Users/ahmad/Library/Python/3.9/lib/python/site-packages/google/api_core/_python_version_support.py:237: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)


In [4]:
# Set file paths
dataset = 'model/urdu_alphabet/keypoint.csv'
model_save_path = 'model/urdu_alphabet/keypoint_classifier.keras'
tflite_save_path = 'model/urdu_alphabet/keypoint_classifier.tflite'

In [5]:
# Set class count
NUM_CLASSES = 38

In [6]:
# Load preprocessed features
# The CSV already contains normalized 42-value landmark vectors, so we load them directly.
X_dataset = np.loadtxt(dataset, delimiter=',', dtype='float32', usecols=list(range(1, (21 * 2) + 1)))

In [7]:
# Load labels
y_dataset = np.loadtxt(dataset, delimiter=',', dtype='int32', usecols=(0))

In [8]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X_dataset, y_dataset, train_size=0.75, random_state=RANDOM_SEED)

In [9]:
# Build classifier
# Input expects the preprocessed 42-value feature vector from the CSV.
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(42,)),  # 21*2 = 42
    
    tf.keras.layers.BatchNormalization(),
    
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    
    tf.keras.layers.Dense(32, activation='relu'),
    
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
])

In [10]:
# Show model summary
model.summary()  # tf.keras.utils.plot_model(model, show_shapes=True)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ batch_normalization             │ (None, 42)             │           168 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         5,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 38)             │         1,254 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,030 (70.43 KB)

 Trainable params: 17,562 (68.60 KB)

 Non-trainable params: 468 (1.83 KB)

In [11]:
# Set callbacks
# Model checkpoint callback
cp_callback = tf.keras.callbacks.ModelCheckpoint(
    model_save_path, verbose=1, save_weights_only=False)
# Callback for early stopping
es_callback = tf.keras.callbacks.EarlyStopping(patience=20, verbose=1)

In [12]:
# Compile model
# Model compilation
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [13]:
# Train model
model.fit(
    X_train,
    y_train,
    epochs=1000,
    batch_size=256,
    validation_data=(X_test, y_test),
    callbacks=[cp_callback, es_callback]
)

Epoch 1/1000
198/207 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.2824 - loss: 2.6854
Epoch 1: saving model to model/urdu_alphabet/keypoint_classifier.keras
207/207 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.2926 - loss: 2.6446 - val_accuracy: 0.7559 - val_loss: 1.2169
Epoch 2/1000
159/207 ━━━━━━━━━━━━━━━━━━━━ 0s 955us/step - accuracy: 0.8105 - loss: 0.6570
Epoch 2: saving model to model/urdu_alphabet/keypoint_classifier.keras
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8172 - loss: 0.6325 - val_accuracy: 0.9475 - val_loss: 0.2614
Epoch 3/1000
163/207 ━━━━━━━━━━━━━━━━━━━━ 0s 937us/step - accuracy: 0.8946 - loss: 0.3676
Epoch 3: saving model to model/urdu_alphabet/keypoint_classifier.keras
207/207 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8971 - loss: 0.3587 - val_accuracy: 0.9734 - val_loss: 0.1182
Epoch 4/1000
159/207 ━━━━━━━━━━━━━━━━━━━━ 0s 958us/step - accuracy: 0.9239 - loss: 0.2692
Epoch 4: saving model to model/urdu_alphabet/keypoint_classifier.keras
207/207

In [14]:
# Evaluate model
# Model evaluation
val_loss, val_acc = model.evaluate(X_test, y_test, batch_size=128)

138/138 ━━━━━━━━━━━━━━━━━━━━ 0s 514us/step - accuracy: 0.9941 - loss: 0.0239


In [15]:
# Reload model
# Loading the saved model
model = tf.keras.models.load_model(model_save_path)

In [17]:
# Save inference model
# Save as a model dedicated to inference
model.save(model_save_path, include_optimizer=False)

In [18]:
# Quantize model
# Transform model (quantization)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quantized_model = converter.convert()

open(tflite_save_path, 'wb').write(tflite_quantized_model)

INFO:tensorflow:Assets written to: /var/folders/0m/xc5njr7x5wd30_5m8c259wr40000gn/T/tmps008l6ux/assets


INFO:tensorflow:Assets written to: /var/folders/0m/xc5njr7x5wd30_5m8c259wr40000gn/T/tmps008l6ux/assets


Saved artifact at '/var/folders/0m/xc5njr7x5wd30_5m8c259wr40000gn/T/tmps008l6ux'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 42), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 38), dtype=tf.float32, name=None)
Captures:
  14368012240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14376569840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4690746000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14390822336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14391291568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14390824448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14385136032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14385136208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14391280912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14397182192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14385221120: TensorS

W0000 00:00:1772477497.890968   16595 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1772477497.891161   16595 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2026-03-02 23:51:37.891313: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/0m/xc5njr7x5wd30_5m8c259wr40000gn/T/tmps008l6ux
2026-03-02 23:51:37.891725: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-03-02 23:51:37.891729: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/0m/xc5njr7x5wd30_5m8c259wr40000gn/T/tmps008l6ux
I0000 00:00:1772477497.896099   16595 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled
2026-03-02 23:51:37.896963: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-03-02 23:51:37.927989: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /var/folder

24088

In [19]:
# Load TFLite model
interpreter = tf.lite.Interpreter(model_path=tflite_save_path)
interpreter.allocate_tensors()

/Users/ahmad/Library/Python/3.9/lib/python/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [20]:
# Read tensor shapes
# Get I / O tensor
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

In [21]:
# Set input sample
interpreter.set_tensor(input_details[0]['index'], np.array([X_test[0]]))

In [22]:
%%time
# Run TFLite inference
# Inference implementation
interpreter.invoke()
tflite_results = interpreter.get_tensor(output_details[0]['index'])

CPU times: user 562 µs, sys: 1.22 ms, total: 1.79 ms
Wall time: 1.71 ms


In [23]:
# Print predictions
print(np.squeeze(tflite_results))
print(np.argmax(np.squeeze(tflite_results)))

[2.5517111e-05 7.3160461e-10 3.1979690e-09 5.7071264e-11 3.8536282e-08
 1.4540694e-07 8.7572893e-08 7.6516510e-10 1.5796098e-10 1.1890454e-10
 1.1990822e-10 3.8918052e-10 7.9303916e-08 4.2131633e-05 7.8857830e-04
 4.8391964e-08 2.3627717e-08 9.9076462e-01 8.3633717e-03 9.9368367e-07
 3.3791184e-10 5.6998792e-06 1.0850941e-06 6.9100547e-06 1.6447860e-09
 4.8254147e-09 3.9552018e-11 1.4551646e-08 7.9605784e-12 2.9782234e-07
 1.8707559e-10 7.0553728e-12 3.5882360e-07 2.6936944e-09 7.9187341e-09
 7.2952949e-11 1.7239510e-13 8.2752919e-12]
17
